# Mathematics Applications

Use this notebook when the **graph family itself** is the object you want to study.

Unlike the physics notebook, you do not need to start from a Hamiltonian or a sampled
surface. You can construct a graph directly, compute its Yamada polynomial, compare
families, and generate exact datasets.

A new user can follow this path:

1. construct one graph and inspect it;
2. compute its Yamada polynomial;
3. compare two evaluation routes;
4. scan one graph family over a parameter;
5. inspect a structured catalog case; and
6. generate a larger dataset only after the single-graph workflow is clear.

The key distinction is:

- **graph construction** specifies the mathematical object;
- **embedded/projection workflows** include geometric crossing information;
- **abstract graph-family calculations** study the combinatorial family directly.

## 1. Set up the mathematical examples

Run these cells once to import SymPy, NetworkX, the Yamada routines, and the structured
graph builders used below.

In [ ]:
from pathlib import Path
import sys
import importlib.util
import os
import tempfile

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DOC_ROOT = PROJECT_ROOT / "doc"
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "knottedgraph-mpl"))

print("project paths configured")
for package in ["numpy", "networkx", "sympy", "plotly", "matplotlib", "pyvista"]:
    print(f"{package:10s} = {importlib.util.find_spec(package) is not None}")


In [ ]:
import math
import time

import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import sympy as sp
from IPython.display import Math, display

from knotted_graph.projection import (
    compute_yamada_polynomial,
    sample_projections,
    select_projection,
)
from knotted_graph.visualization import plot_3D_graph_plotly

BLUE = "#1f77b4"
RED = "#d62728"
CAMERA = dict(eye=dict(x=4.0, y=4.0, z=3.0))
pio.renderers.default = "notebook_connected"
Y = sp.Symbol("Y")
kx, ky, kz = sp.symbols("k_x k_y k_z", real=True)


def axis_style():
    return dict(
        visible=True,
        title="",
        showticklabels=False,
        showbackground=False,
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor="black",
        linewidth=2,
    )


def apply_kg_layout(fig, *, width=760, height=620):
    fig.update_layout(
        title=None,
        width=width,
        height=height,
        margin=dict(l=0, r=0, t=0, b=0),
        scene=dict(
            xaxis=axis_style(),
            yaxis=axis_style(),
            zaxis=axis_style(),
            aspectmode="data",
            camera=CAMERA,
        ),
    )
    return fig


def plot_surface_polydata(surface, *, opacity=0.58):
    mesh = surface.triangulate()
    faces = mesh.faces.reshape(-1, 4)[:, 1:]
    pts = mesh.points
    fig = go.Figure(
        go.Mesh3d(
            x=pts[:, 0],
            y=pts[:, 1],
            z=pts[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=BLUE,
            opacity=opacity,
        )
    )
    return apply_kg_layout(fig)


def plot_points_3d(points, *, size=3):
    points = np.asarray(points)
    fig = go.Figure(
        go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode="markers",
            marker=dict(size=size, color=BLUE),
        )
    )
    return apply_kg_layout(fig)


def plot_graph_kg(graph):
    return apply_kg_layout(plot_3D_graph_plotly(graph))


def print_upsilon(label, expr):
    print(f"Upsilon({label}; Y) = {sp.expand(expr)}")


def display_bloch_vector(label, components):
    display(Math(label + r"=" + sp.latex(sp.Matrix(components))))


print("shared plotting and notation helpers ready")

from plotly.subplots import make_subplots


def add_surface_trace(
    fig,
    surface,
    *,
    row=1,
    col=1,
    opacity=0.58,
    color=BLUE,
):
    mesh = surface.triangulate()
    faces = mesh.faces.reshape(-1, 4)[:, 1:]
    pts = np.asarray(mesh.points)

    fig.add_trace(
        go.Mesh3d(
            x=pts[:, 0],
            y=pts[:, 1],
            z=pts[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=color,
            opacity=opacity,
            showscale=False,
        ),
        row=row,
        col=col,
    )


def add_graph_traces(
    fig,
    graph,
    *,
    row=1,
    col=1,
):
    graph_fig = (
        plot_3D_graph_plotly(
            graph
        )
    )

    for trace in graph_fig.data:
        fig.add_trace(
            trace,
            row=row,
            col=col,
        )


def style_plotly_scenes(
    fig,
    scene_count,
    *,
    width=980,
    height=660,
):
    for index in range(
        1,
        scene_count + 1,
    ):
        scene_name = (
            "scene"
            if index == 1
            else f"scene{index}"
        )

        fig.update_layout(
            **{
                scene_name: dict(
                    xaxis=axis_style(),
                    yaxis=axis_style(),
                    zaxis=axis_style(),
                    aspectmode="data",
                    camera=CAMERA,
                )
            }
        )

    fig.update_layout(
        width=width,
        height=height,
        margin=dict(
            l=0,
            r=0,
            t=10,
            b=0,
        ),
        showlegend=False,
    )

    return fig


def surface_component_summary(
    surface,
):
    try:
        bodies = (
            surface.split_bodies()
        )

        components = [
            body
            for body in bodies
            if (
                body is not None
                and getattr(
                    body,
                    "n_cells",
                    0,
                ) > 0
            )
        ]

        sizes = sorted(
            [
                (
                    component.n_points,
                    component.n_cells,
                )
                for component
                in components
            ],
            key=lambda item: item[1],
            reverse=True,
        )

        return (
            len(components),
            sizes,
        )

    except Exception:
        return None, []


def print_surface_summary(
    label,
    surface,
):
    n_components, sizes = (
        surface_component_summary(
            surface
        )
    )

    print(
        f"{label}: "
        f"{surface.n_points} points, "
        f"{surface.n_cells} cells"
    )

    if n_components is None:
        print(
            "  connected components = unavailable"
        )

    else:
        print(
            "  connected components =",
            n_components,
        )

        if n_components > 1:
            print(
                "  component sizes "
                "(points, cells) =",
                sizes,
            )

## 2. Compare two Yamada evaluation routes

When the library provides more than one exact evaluation route, comparing them on the
same graph is a useful consistency check.

The two expressions should agree after simplification. If they do not, inspect the graph
construction and the method assumptions before scanning a larger family.

In [ ]:
def mathematical_k4_spine(samples=90, amplitude=0.75):
    vertices = {
        "a": np.array([-1.15, -0.78, -0.38]),
        "b": np.array([1.18, -0.64, 0.30]),
        "c": np.array([0.86, 0.95, -0.26]),
        "d": np.array([-0.88, 0.84, 0.52]),
    }
    edge_specs = [
        ("a", "b", "ab", np.array([0.00, 0.90, 0.70]), 0.0),
        ("a", "c", "ac", np.array([0.35, -0.15, 1.00]), 1.1),
        ("a", "d", "ad", np.array([0.95, 0.15, -0.25]), 2.2),
        ("b", "c", "bc", np.array([-0.90, 0.25, 0.35]), 0.7),
        ("b", "d", "bd", np.array([-0.20, 1.00, -0.60]), 1.7),
        ("c", "d", "cd", np.array([0.10, -0.90, -0.85]), 2.8),
    ]
    s = np.linspace(0.0, 1.0, samples)
    graph = nx.MultiGraph()
    for vertex_id, pos in vertices.items():
        graph.add_node(vertex_id, pos=pos.copy())

    for u, v, key, bend, phase in edge_specs:
        start = vertices[u]
        end = vertices[v]
        chord = end - start
        bend = bend / np.linalg.norm(bend)
        side = np.cross(chord, bend)
        side = side / np.linalg.norm(side)
        envelope = np.sin(np.pi * s)
        pts = (1 - s)[:, None] * start + s[:, None] * end
        pts += amplitude * envelope[:, None] * (
            np.cos(phase + np.pi * s)[:, None] * bend
            + 0.6 * np.sin(2 * np.pi * s + phase)[:, None] * side
        )
        pts[0] = start
        pts[-1] = end
        graph.add_edge(u, v, key=key, pts=pts)

    graph.graph.update(
        graph_id="mathematical_k4",
        input_kind="internal_mathematical_geometry",
        is_closed=True,
    )
    return graph


math_graph = mathematical_k4_spine()
math_projection = select_projection(math_graph, num_rotation_samples=16)

negami = compute_yamada_polynomial(
    math_graph,
    Y,
    rotation_angles=math_projection.rotation_angles,
    method="negami",
    n_jobs=1,
)
recursive = compute_yamada_polynomial(
    math_graph,
    Y,
    rotation_angles=math_projection.rotation_angles,
    method="recursive",
    n_jobs=1,
)
print_upsilon("G_K4, negami", negami)
print_upsilon("G_K4, recursive", recursive)
print("same_result =", sp.expand(negami - recursive) == 0)


## 3. Scan the theta-graph family

A family scan asks how the invariant changes as a discrete graph parameter changes.

The code below builds several $\Theta_s$ graphs, computes the polynomial for each one,
and keeps the parameter value beside the result. Change the scan range to explore a
larger family.

In [ ]:
from knotted_graph.core import ThetaGraph
from knotted_graph.invariants.yamada import compute_yamada_polynomial_recursive

for s in range(2, 8):
    theta = ThetaGraph(s)
    upsilon = compute_yamada_polynomial_recursive(theta, Y)
    print_upsilon(f"Theta_{s}", upsilon)


## 4. Explore the structured graph catalog

The catalog provides reusable constructors for several graph families.

Start by listing the available family names. Then build one case, inspect its graph
summary and plot, and compute the invariant. This is the easiest way to learn the
parameter convention for a family before launching a large sweep.

In [ ]:
from knotted_graph.applications.mathematical import (
    GRAPH_FAMILY_CATALOG,
    NOTEBOOK_YAMADA_EXAMPLES,
    build_graph_case,
    graph_summary,
    plot_structured_multigraph,
)
from knotted_graph.invariants.yamada import laurent_y_to_sigma_polynomial

sigma = sp.Symbol("sigma")

for family_name, spec in GRAPH_FAMILY_CATALOG.items():
    print(f"{family_name:24s} sample_args={spec.sample_args}  {spec.note}")


### 4.1 Compute and compare polynomial representations

For each catalog example, the next cells record:

- a compact graph summary;
- the Laurent polynomial in `Y`; and
- the converted polynomial in `sigma`.

The plots keep loops and parallel edges visible so you can compare the graph
structure directly with its exact invariant.

In [ ]:
catalog_results = {}
cols = 3
rows = math.ceil(len(NOTEBOOK_YAMADA_EXAMPLES) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
axes = np.asarray(axes).reshape(-1)

for ax, (family_name, args, label) in zip(axes, NOTEBOOK_YAMADA_EXAMPLES):
    graph, pos = build_graph_case(family_name, *args)
    plot_structured_multigraph(
        graph,
        pos,
        family_name=family_name,
        family_args=args,
        ax=ax,
        show=False,
        node_color=RED,
        node_edge_color=RED,
        edge_color=BLUE,
    )
    yamada_y = sp.expand(compute_yamada_polynomial_recursive(graph, Y))
    yamada_sigma = laurent_y_to_sigma_polynomial(yamada_y, Y, sigma).as_expr()
    catalog_results[label] = {
        "summary": graph_summary(graph),
        "Y": yamada_y,
        "sigma": sp.expand(yamada_sigma),
    }

for ax in axes[len(NOTEBOOK_YAMADA_EXAMPLES):]:
    ax.set_axis_off()

plt.tight_layout()
plt.show()

for panel_index, label in enumerate(catalog_results, start=1):
    result = catalog_results[label]
    print(f"panel {panel_index}: {label}  {result['summary']}")
    print_upsilon(label, result["Y"])
    print(f"Upsilon_sigma({label}; sigma) = {result['sigma']}")
    print()


### 4.2 Scan a parameterized family

Call the same graph-family builder repeatedly when you want to study how topology
changes with a discrete parameter.

The cylinder example below varies one parameter and stores the resulting
polynomials. You can use the same pattern for your own family by replacing the
builder and parameter range.

In [ ]:
cylinder_scan = []
for cols in range(3, 7):
    graph, pos = build_graph_case("cylinder", 2, cols)
    yamada_y = sp.expand(compute_yamada_polynomial_recursive(graph, Y))
    yamada_sigma = laurent_y_to_sigma_polynomial(yamada_y, Y, sigma).as_expr()
    cylinder_scan.append(
        (cols, graph_summary(graph), yamada_y, sp.expand(yamada_sigma))
    )

graph, pos = build_graph_case("cylinder", 2, 6)
plot_structured_multigraph(
    graph,
    pos,
    family_name="cylinder",
    family_args=(2, 6),
    node_color=RED,
    node_edge_color=RED,
    edge_color=BLUE,
)

for cols, summary, yamada_y, yamada_sigma in cylinder_scan:
    label = f"Cylinder(2,{cols})"
    print(f"{label}: {summary}")
    print_upsilon(label, yamada_y)
    print(f"Upsilon_sigma({label}; sigma) = {yamada_sigma}")
    print()


## 5. Inspect any catalog case interactively

`inspect_catalog_case(...)` is a small convenience wrapper around the public catalog
builder and plotting functions.

Pass the family name followed by that family's parameters. The helper then:

1. builds the graph;
2. plots it;
3. prints basic graph information;
4. computes $\Upsilon(G;Y)$; and
5. prints the corresponding $\sigma$ representation.

Use it to verify a few cases manually before generating a dataset.

In [ ]:
def inspect_catalog_case(family_name, *args):
    graph, pos = build_graph_case(family_name, *args)
    plot_structured_multigraph(
        graph,
        pos,
        family_name=family_name,
        family_args=args,
        node_color=RED,
        node_edge_color=RED,
        edge_color=BLUE,
    )
    yamada_y = sp.expand(compute_yamada_polynomial_recursive(graph, Y))
    yamada_sigma = sp.expand(
        laurent_y_to_sigma_polynomial(yamada_y, Y, sigma).as_expr()
    )
    print("summary =", graph_summary(graph))
    print_upsilon(f"{family_name}{args}", yamada_y)
    print(f"Upsilon_sigma({family_name}{args}; sigma) = {yamada_sigma}")
    return graph, yamada_y, yamada_sigma


# Example:
inspect_catalog_case("cylinder", 2, 5)


## 6. Test a Petersen-minor certificate for intrinsic linkedness

Graph-minor questions belong naturally in the mathematical workflow because they concern
the **abstract graph**, rather than the physical field or surface that produced a
particular embedding.

The example below starts from the extracted “Awesome” graph and asks whether it contains
the Petersen graph as a minor.

Interpret the result as follows:

- `True` gives a Petersen-minor certificate. Since the Petersen graph belongs to the
  Petersen family of forbidden minors for linkless embeddings, this is sufficient to
  establish intrinsic linkedness of the abstract graph.
- `False` does **not** establish linkless embeddability. It only says that this particular
  Petersen target was not found; other Petersen-family minors would still need to be
  checked for a complete forbidden-minor test.

The code also returns the branch sets used for the detected minor, when available.

The built-in nodal constructors define a two-band non-Hermitian Hamiltonian from a
complex polynomial $f(z,w)$.  They first return the Bloch vector

$$
\mathbf d_f(\mathbf k)
=
\left(
\operatorname{Re}f,\,
i\gamma,\,
\operatorname{Im}f
\right),
$$

and `NodalSkeleton` constructs

$$
H_f(\mathbf k;\gamma)
=
\mathbf d_f(\mathbf k)\cdot\boldsymbol{\sigma}
=
\begin{pmatrix}
\operatorname{Im}f & \operatorname{Re}f+\gamma\\
\operatorname{Re}f-\gamma & -\operatorname{Im}f
\end{pmatrix}.
$$

Unless a model explicitly overrides them,

$$
z
=
\cos(2k_z)+c
+i\left(
\cos k_x+\cos k_y+\cos k_z-m
\right),
\qquad
w
=
\sin k_x+i\sin k_y.
$$

For the “Awesome” model,

$$
f_{\mathrm{Awesome}}(z,w)
=
z\left(z^2-w^4+w\right).
$$

The extracted graph in this example uses $\gamma=0.2$ and $c=0.5$ before the abstract
minor test is applied.

In [ ]:
from knotted_graph.applications.nodal import NodalSkeleton
from knotted_graph.applications.nodal.models import (
    awesome_bloch_vector,
    threelink_bloch_vector,
)


awesome_minor_ske = NodalSkeleton(
    awesome_bloch_vector(
        0.2,
        k_symbols=(kx, ky, kz),
        c=0.5,
    ),
    k_symbols=(kx, ky, kz),
    dimension=300,
    axis_scale=(1.0, 1.0, 1.5),
)

awesome_minor_graph = (
    awesome_minor_ske.skeleton_graph(
        simplify=True,
        smooth_epsilon=2,
    )
)

petersen_graph = nx.petersen_graph()

minor_embedding = (
    awesome_minor_ske.check_minor(
        petersen_graph,
        awesome_minor_graph,
    )
)

print(
    "awesome graph nodes / edges =",
    (
        awesome_minor_graph.number_of_nodes(),
        awesome_minor_graph.number_of_edges(),
    ),
)
print(
    "degree sequence =",
    sorted(
        dict(
            awesome_minor_graph.degree()
        ).values()
    ),
)
print(
    "Petersen minor found =",
    bool(minor_embedding),
)

if minor_embedding:
    print(
        "branch-set sizes =",
        {
            node: len(branch_set)
            for node, branch_set
            in minor_embedding.items()
        },
    )


# Inspect the extracted embedded graph.
plot_graph_kg(
    awesome_minor_graph
).show()


# Inspect the target Petersen graph separately.
fig, ax = plt.subplots(
    figsize=(5.4, 5.0)
)

petersen_pos = nx.spring_layout(
    petersen_graph,
    seed=4,
)

nx.draw_networkx_edges(
    petersen_graph,
    pos=petersen_pos,
    ax=ax,
    edge_color=BLUE,
    width=2.0,
)

nx.draw_networkx_nodes(
    petersen_graph,
    pos=petersen_pos,
    ax=ax,
    node_color=RED,
    node_size=180,
)

ax.set_title(
    "Target Petersen graph"
)
ax.set_aspect("equal")
ax.axis("off")
plt.show()

## 7. Track planarity and connectivity during a parameter sweep

After the intrinsic-linkedness example, it is natural to inspect additional
**abstract graph diagnostics** across a parameter family.

For each parameter value, this example:

1. keeps the complete extracted surface;
2. extracts the embedded spatial graph;
3. records graph connectivity information; and
4. tests whether the underlying abstract graph is planar.

The distinction is important:

- **graph connectivity** concerns how the abstract graph is connected;
- **abstract planarity** asks whether that graph can be drawn in the plane without
  crossings;
- the topology of the specific embedding in $\mathbb{R}^3$ is a separate question.

Therefore a planar abstract graph is not automatically a topologically trivial
three-dimensional embedding.

The built-in nodal constructors define a two-band non-Hermitian Hamiltonian from a
complex polynomial $f(z,w)$.  They first return the Bloch vector

$$
\mathbf d_f(\mathbf k)
=
\left(
\operatorname{Re}f,\,
i\gamma,\,
\operatorname{Im}f
\right),
$$

and `NodalSkeleton` constructs

$$
H_f(\mathbf k;\gamma)
=
\mathbf d_f(\mathbf k)\cdot\boldsymbol{\sigma}
=
\begin{pmatrix}
\operatorname{Im}f & \operatorname{Re}f+\gamma\\
\operatorname{Re}f-\gamma & -\operatorname{Im}f
\end{pmatrix}.
$$

Unless a model explicitly overrides them,

$$
z
=
\cos(2k_z)+c
+i\left(
\cos k_x+\cos k_y+\cos k_z-m
\right),
\qquad
w
=
\sin k_x+i\sin k_y.
$$

For the three-link model,

$$
f_{\mathrm{3L}}(z,w)
=
z\left(z^2-w^2\right),
\qquad
c=0.5,\quad m=2.
$$

In [ ]:
three_link_planarity_records = []
for gamma in (0.116, 0.41, 0.5):
    ske_three = NodalSkeleton(
        threelink_bloch_vector(gamma, k_symbols=(kx, ky, kz)),
        k_symbols=(kx, ky, kz),
        dimension=300,
        axis_scale=(1.0, 1.0, 1.5),
    )
    surface_three = ske_three.exceptional_surface_pv
    graph_three = ske_three.skeleton_graph(simplify=True, smooth_epsilon=2)
    planar_three = nx.check_planarity(nx.Graph(graph_three))[0]
    three_link_planarity_records.append(
        (gamma, surface_three, graph_three, planar_three)
    )
    print(f"gamma = {gamma}")
    print_surface_summary("  full surface", surface_three)
    print(
        "  graph_nodes_edges =",
        (graph_three.number_of_nodes(), graph_three.number_of_edges()),
    )
    print("  planar =", planar_three)


In [ ]:
fig = make_subplots(
    rows=2,
    cols=3,
    specs=[
        [{"type": "scene"} for _ in range(3)],
        [{"type": "scene"} for _ in range(3)],
    ],
    horizontal_spacing=0.01,
    vertical_spacing=0.02,
)
for col, (
    gamma,
    surface_three,
    graph_three,
    planar_three,
) in enumerate(three_link_planarity_records, start=1):
    add_surface_trace(fig, surface_three, row=1, col=col, opacity=0.46)
    add_graph_traces(fig, graph_three, row=2, col=col)
style_plotly_scenes(fig, 6, width=1080, height=720).show()


## 8. Generate a small example Yamada dataset

Dataset generation can become expensive very quickly because exact Yamada calculations
grow with graph complexity.

For the user guide, the live calculation is therefore intentionally **small**: only six
simple catalog cases are evaluated. Its purpose is to show the dataset-generation
workflow without making a new user wait for a large exact-computation sweep.

Each generated row stores:

- `graph_name`;
- `varying_params`;
- `yamada`;
- `yamada_sigma`.

A larger precomputed dataset is supplied as
`structured_graph_yamada_dataset.csv` in the **same folder as this notebook** and is
loaded in the next section.

In [12]:
import csv


# Six deliberately small cases for a fast tutorial run.
DEMO_YAMADA_SWEEPS = {
    "periodic_theta": [
        (2,),
        (3,),
    ],
    "fan": [
        (2,),
        (3,),
    ],
    "ladder": [
        (2,),
        (3,),
    ],
}

planned_demo_row_count = sum(
    len(args_list)
    for args_list in DEMO_YAMADA_SWEEPS.values()
)


def compute_demo_yamada_dataset(output_path=None):
    """Compute a small exact dataset suitable for a quick tutorial run."""
    fieldnames = [
        "graph_name",
        "varying_params",
        "yamada",
        "yamada_sigma",
    ]

    rows = []

    print(
        "Planned tutorial rows =",
        planned_demo_row_count,
    )

    for family_name, args_list in DEMO_YAMADA_SWEEPS.items():
        builder = (
            GRAPH_FAMILY_CATALOG[
                family_name
            ].builder
        )

        for args in args_list:
            graph = builder(*args)

            yamada_expr = sp.expand(
                compute_yamada_polynomial_recursive(
                    graph,
                    Y,
                )
            )

            yamada_sigma_expr = (
                laurent_y_to_sigma_polynomial(
                    yamada_expr,
                    Y,
                    sigma,
                )
                .as_expr()
            )

            row = {
                "graph_name": family_name,
                "varying_params": repr(
                    tuple(args)
                ),
                "yamada": sp.sstr(
                    yamada_expr
                ),
                "yamada_sigma": sp.sstr(
                    yamada_sigma_expr
                ),
            }

            rows.append(row)

            print(
                f"{family_name:18s} "
                f"{row['varying_params']:8s} "
                f"{row['yamada']}"
            )

    if output_path is not None:
        output_path = Path(
            output_path
        )

        with output_path.open(
            "w",
            newline="",
            encoding="utf-8",
        ) as handle:
            writer = csv.DictWriter(
                handle,
                fieldnames=fieldnames,
            )
            writer.writeheader()
            writer.writerows(rows)

        print(
            f"\nSaved demo dataset to "
            f"{output_path}"
        )

    return rows

### 8.1 Run the small demonstration

This cell computes only six small graph cases, so it is suitable for a first tutorial
run.

The demo is intentionally separate from the supplied full CSV. Running this cell will
not overwrite `structured_graph_yamada_dataset.csv`.

In [ ]:
demo_yamada_dataset = (
    compute_demo_yamada_dataset()
)

print(
    f"\nComputed "
    f"{len(demo_yamada_dataset)} "
    f"tutorial rows."
)

demo_yamada_dataset

## 9. Inspect the supplied structured-graph Yamada dataset

For broader coverage, use the supplied
`structured_graph_yamada_dataset.csv` rather than recomputing a large exact dataset
during the tutorial.

The CSV is distributed in the **same `applications/` folder** as this notebook. It
contains the precomputed columns

- `graph_name`;
- `varying_params`;
- `yamada`; and
- `yamada_sigma`.

The cell below locates that file, reports its size, and displays the **complete CSV in a
scrollable table**.

In [ ]:
import csv
import html
from IPython.display import HTML


def find_local_structured_dataset():
    """Locate the CSV distributed beside this notebook."""
    filename = (
        "structured_graph_yamada_dataset.csv"
    )

    candidates = [
        Path.cwd() / filename,
        PROJECT_ROOT
        / "doc"
        / "user_guide"
        / "applications"
        / filename,
        PROJECT_ROOT
        / "doc"
        / "applications"
        / filename,
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    # Fallback for a differently named documentation folder.
    matches = list(
        PROJECT_ROOT.glob(
            f"**/applications/{filename}"
        )
    )

    if len(matches) == 1:
        return matches[0].resolve()

    if len(matches) > 1:
        raise FileNotFoundError(
            "More than one structured_graph_yamada_dataset.csv "
            "was found. Run this notebook from its applications "
            "folder so the intended file is unambiguous."
        )

    raise FileNotFoundError(
        "Could not find structured_graph_yamada_dataset.csv. "
        "Keep the CSV in the same applications folder as "
        "02_mathematics_applications.ipynb."
    )


dataset_path = (
    find_local_structured_dataset()
)

with dataset_path.open(
    newline="",
    encoding="utf-8",
) as handle:
    reader = csv.DictReader(
        handle
    )
    stored_rows = list(
        reader
    )
    stored_columns = list(
        reader.fieldnames or []
    )

print(
    "dataset path =",
    dataset_path,
)
print(
    "stored rows =",
    len(stored_rows),
)
print(
    "columns =",
    stored_columns,
)


def _csv_html_table(
    rows,
    columns,
):
    header = "".join(
        f"<th>{html.escape(column)}</th>"
        for column in columns
    )

    body_rows = []

    for row in rows:
        cells = "".join(
            "<td style='vertical-align:top;"
            "padding:4px 8px;"
            "white-space:pre-wrap;'>"
            f"{html.escape(str(row.get(column, '')))}"
            "</td>"
            for column in columns
        )

        body_rows.append(
            f"<tr>{cells}</tr>"
        )

    return f"""
    <div style="
        max-height:620px;
        overflow:auto;
        border:1px solid #ddd;
    ">
      <table style="
          border-collapse:collapse;
          width:100%;
          font-family:monospace;
          font-size:12px;
      ">
        <thead style="
            position:sticky;
            top:0;
            background:white;
            z-index:1;
        ">
          <tr>{header}</tr>
        </thead>
        <tbody>
          {''.join(body_rows)}
        </tbody>
      </table>
    </div>
    """


display(
    HTML(
        _csv_html_table(
            stored_rows,
            stored_columns,
        )
    )
)

## 10. Use exact computation to discover a mathematical pattern

A productive workflow for a new graph family is:

1. define the family and its discrete parameters;
2. compute exact invariants over the largest practical range;
3. inspect degrees, factors, coefficients, and possible recurrences;
4. formulate a conjecture from the exact data; and
5. prove or independently verify the recurrence or closed form.

Finite computation gives evidence and candidate structure. It does not replace a proof.

For projection-level diagnostics, state expansions, and geometric robustness tests, continue to
**[Advanced & Reproduction](../03_advanced_and_reproduction.ipynb)**.